In [2]:
#   Install Unsloth library
! pip install unsloth --upgrade --no-cache-dir -q

!pip install --no-deps xformers trl peft accelerate bitsandbytes triton wandb

In [3]:
import unsloth
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only

import torch

from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

import wandb

import os

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
from kaggle_secrets import UserSecretsClient
wab_token = UserSecretsClient().get_secret("wandb_api_key")

In [5]:
# 2. Log in to Weights & Biases
wandb.login(key=wab_token)

# 3. Initialize your W&B project (this creates the project if it doesn't exist yet)
wandb.init(
    project="gemma-2-2b-tool-calling",  # Name of your project on W&B
    name="unsloth-t4-run",            # Name of this specific training run (optional)
    config={                            # Optional: log your hyperparameters for tracking
        "model": "gemma-2-2b-it",
        "quantization": "4-bit",
        "max_seq_length": 2048,
        "learning_rate": 2e-4,
    }
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: djzaamir to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [6]:
max_seq_len = 2048
model_id = "unsloth/gemma-2-2b-it-bnb-4bit"
model, tokenizer =FastLanguageModel.from_pretrained(
    model_name = model_id,
    max_seq_length = max_seq_len,
    dtype = None,        # None for auto detection.
    load_in_4bit = True 
)

==((====))==  Unsloth 2026.10.11: Fast Gemma2 patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Unsloth: Will load unsloth/gemma-2-2b-it-bnb-4bit as a legacy tokenizer.


In [7]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0. Suggested: 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, # Optimized to 0 by Unsloth
    bias = "none",    # Optimized to "none" by Unsloth
    use_gradient_checkpointing = "unsloth", # Saves VRAM / supports long contexts
    random_state = 42,
    max_seq_length = max_seq_len,
)

Unsloth 2026.10.11 patched 26 layers with 26 QKV layers, 26 O layers and 26 MLP layers.


Unsloth: compiling each transformer block as one region (Gemma2DecoderLayer).


In [8]:
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma2", # Map to Hermes ChatML format
)

[unsloth.chat_templates|WARNING]Unsloth: Will map <end_of_turn> to EOS = <eos>.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/_unsloth_sentencepiece_temp/tokenizer_nw7yfv_s/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /kaggle/working/_unsloth_sentencepiece_temp/tokenizer_nw7yfv_s.


In [9]:
dataset = load_dataset("NousResearch/hermes-function-calling-v1", "func_calling")

README.md:   0%|          | 0.00/37.7k [00:00<?, ?B/s]

func-calling.json: reconstructing file:   0%|          |  0.00B / 20.6MB            

func-calling.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1893 [00:00<?, ? examples/s]

In [10]:
dataset = dataset["train"].train_test_split(
    test_size=0.2,
    seed=42
)
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'conversations', 'tools', 'category', 'subcategory', 'task'],
        num_rows: 1514
    })
    test: Dataset({
        features: ['id', 'conversations', 'tools', 'category', 'subcategory', 'task'],
        num_rows: 379
    })
})

In [11]:
for conv in dataset["train"][0]["conversations"]:
    print(conv)
    print("="*30)

{'from': 'system', 'value': 'You are an expert structured information extraction AI model. You will be provided with documents to extract information from. You are also provided with the json schema to output extracted information in the function signatures within XML tags <tools></tools>. Don\'t make assumptions about what values to plug into json schema. \n<tools>\n[{"type": "function", "function": {"name": "ExpertQAExtractor", "description": "Extracts a list of questions that request for an opinion on a topic found in the document.", "parameters": {"type": "object", "properties": {"topic_opinion_questions": {"type": "array", "items": {"type": "string"}}}, "required": ["topic_opinion_questions"]}}}]\n</tools>\nFor each extraction function call return a json object with function name and arguments followed by a <tool_call> tag with the following schema:\n<tool_call>\n{"name": <function-name>, "arguments": <args-dict>}\n</tool_call>'}
{'from': 'human', 'value': 'Can you help me extract

In [12]:
def convert_hermes_format(examples):
    converted_conversations = []
    for convo in examples["conversations"]:
        new_convo = []
        for message in convo:
            # Get the original role from 'from' or 'role'
            role = message.get("from", message.get("role", ""))
            
            # Map legacy ShareGPT roles to standard chat roles
            if role == "human" or role == "system": # Gemma2 does not support `system` roles it only supports user or assistant
                role = "user"
            elif role == "gpt":
                role = "assistant"
            else: # If it is anything else then also make it a user, because gemma-2-it models only work with `user` and `assistant` roles
                role = "user"
                
            # 'tool' is kept as-is or handled based on your template
            
            new_convo.append({
                "role": role,
                "content": message.get("value", message.get("content", ""))
            })
        converted_conversations.append(new_convo)
    
    return {"conversations": converted_conversations}


In [13]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'conversations', 'tools', 'category', 'subcategory', 'task'],
        num_rows: 1514
    })
    test: Dataset({
        features: ['id', 'conversations', 'tools', 'category', 'subcategory', 'task'],
        num_rows: 379
    })
})

In [14]:
# Apply your custom mapper instead of standardize_sharegpt
dataset_train = dataset["train"].map(convert_hermes_format, batched=True)
dataset_test = dataset["test"].map(convert_hermes_format, batched=True)

Map:   0%|          | 0/1514 [00:00<?, ? examples/s]

Map:   0%|          | 0/379 [00:00<?, ? examples/s]

In [15]:
print(dataset_train)
print("============="*7)
print(dataset_test)

Dataset({
    features: ['id', 'conversations', 'tools', 'category', 'subcategory', 'task'],
    num_rows: 1514
})
Dataset({
    features: ['id', 'conversations', 'tools', 'category', 'subcategory', 'task'],
    num_rows: 379
})


In [16]:
# Now the roles which were given by the hermes dataset will be mapped to the roles expected by the gemma-2 models
for conv in dataset_train[0]["conversations"]:
    print(conv)
    print("="*30)

{'content': 'You are an expert structured information extraction AI model. You will be provided with documents to extract information from. You are also provided with the json schema to output extracted information in the function signatures within XML tags <tools></tools>. Don\'t make assumptions about what values to plug into json schema. \n<tools>\n[{"type": "function", "function": {"name": "ExpertQAExtractor", "description": "Extracts a list of questions that request for an opinion on a topic found in the document.", "parameters": {"type": "object", "properties": {"topic_opinion_questions": {"type": "array", "items": {"type": "string"}}}, "required": ["topic_opinion_questions"]}}}]\n</tools>\nFor each extraction function call return a json object with function name and arguments followed by a <tool_call> tag with the following schema:\n<tool_call>\n{"name": <function-name>, "arguments": <args-dict>}\n</tool_call>', 'role': 'user'}
{'content': 'Can you help me extract queries from t

In [17]:
def formatting_prompts_func(examples):
    """Maps raw conversation dictionaries into tokenizable strings matching ChatML syntax."""

    
    convos = examples["conversations"]
    
    # Only taking the first 3 conversations i-e system-prompt, user-prompt, model-response (Which is a tool call)
    # that is why you will see convo[0:3]
    texts = [tokenizer.apply_chat_template(convo[0:3], tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

In [18]:
# Map formatting across the dataset efficiently using multi-processing
dataset_train = dataset_train.map(formatting_prompts_func, batched=True)
dataset_test = dataset_test.map(formatting_prompts_func, batched=True)

Map:   0%|          | 0/1514 [00:00<?, ? examples/s]

Map:   0%|          | 0/379 [00:00<?, ? examples/s]

In [19]:
dataset_train["text"][99]

'<bos><start_of_turn>user\nYou are a function calling AI model. You are provided with function signatures within <tools> </tools> XML tags. You may call one or more functions to assist with the user query. Don\'t make assumptions about what values to plug into functions.\n<tools>\n[{"type": "function", "function": {"name": "calculate_portfolio_VaR", "description": "Calculates the Value at Risk (VaR) for a given energy trading portfolio over a specified time horizon at a certain confidence level.", "parameters": {"type": "object", "properties": {"portfolio_id": {"type": "string", "description": "The unique identifier for the energy trading portfolio."}, "confidence_level": {"type": "number", "description": "The confidence level for the VaR calculation, typically between 0.90 and 0.99."}, "time_horizon": {"type": "number", "description": "The time horizon for the VaR calculation, in days."}}, "required": ["portfolio_id", "confidence_level", "time_horizon"]}}}, {"type": "function", "funct

In [28]:


# Older version of SFTTrainer (works on CPUs though)
# trainer = SFTTrainer(
#     model=model,
#     tokenizer=tokenizer,
#     train_dataset=dataset_train, # Passing the training split
#     eval_dataset=dataset_test, # Test split
#     dataset_text_field="text",
#     max_seq_length=2048,            # Capped at 2048 to fit comfortably within T4 16GB VRAM limits
#     dataset_num_proc=2,             # Parallelizes data preprocessing to cut preparation time.
    
#     # Important to packing False, as we dont want to mix the context of very different tool calls with each other
#     packing=False,                  # Set to True if you want to pack short sequences together (speeds up training 2x).
       
#     args=TrainingArguments(
#         per_device_train_batch_size=2,  # Safe batch size for a 2B 4-bit model at 2048 seq length on a T4
#         gradient_accumulation_steps=4,  # Accumulates gradients to achieve an effective batch size of 8
#         max_steps=500,                  # Adjust based on your dataset size (e.g., total_samples * epochs / effective_batch_size)
#         learning_rate=2e-4,             # Standard, effective learning rate for LoRA fine-tuning
#         fp16=not torch.cuda.is_bf16_supported(), # Automatically enables FP16 on T4 GPUs (T4 lacks native BF16 support).
#         bf16=torch.cuda.is_bf16_supported(),   # Enables BF16 on modern GPUs (A100/L4/H100) for better stability.
#         logging_steps=1,                # Logs metrics to W&B every single step for maximum observability.
#         optim="adamw_8bit",             # 8-bit AdamW optimizer heavily cuts VRAM usage, crucial for T4
#         weight_decay=0.01,              # Regularization penalty to prevent overfitting on complex functions.
#         lr_scheduler_type="cosine",     # Smooth decay curve for learning rate convergence.
#         seed=3407,                      # Unsloth's recommended seed for stable reproducibility
#         output_dir="outputs",           # Directory to save model checkpoints and adapters
#         report_to="wandb",              # Hooks logs directly into Weights & Biases experiment dashboard.
#     ),
# )


trainer = SFTTrainer(
    model=model,
    train_dataset=dataset_train, # Passing the training split
    eval_dataset=dataset_test, # Test split
    
    # Pass dataset configuration and sequence limits via SFTConfig instead
    args=SFTConfig(
        output_dir="outputs",           # Directory to save model checkpoints and adapters
        max_seq_length=2048,            # Capped at 2048 to fit comfortably within T4 16GB VRAM limits
        packing=False,                  # Set to True if you want to pack short sequences together
        
        per_device_train_batch_size=4,  # Safe batch size for a 2B 4-bit model at 2048 seq length on a T4
        gradient_accumulation_steps=4,  # Accumulates gradients to achieve an effective batch size of 8
        max_steps=500,                  # Adjust based on your dataset size
        learning_rate=2e-4,             # Standard, effective learning rate for LoRA fine-tuning
        fp16=not torch.cuda.is_bf16_supported(), # Automatically enables FP16 on T4 GPUs
        bf16=torch.cuda.is_bf16_supported(),   # Enables BF16 on modern GPUs (A100/L4/H100)
        logging_steps=1,                # Logs metrics to W&B every single step
        
        eval_strategy="steps",          # Evaluate periodically
        eval_steps=50,                  # Evaluate every 50 steps
        save_strategy="steps",
        save_steps=50,
        
        optim="adamw_8bit",             # 8-bit AdamW optimizer heavily cuts VRAM usage
        weight_decay=0.01,              # Regularization penalty to prevent overfitting
        lr_scheduler_type="cosine",     # Smooth decay curve for learning rate convergence
        seed=3407,                      # Unsloth's recommended seed for stable reproducibility
        report_to="wandb",              # Hooks logs directly into Weights & Biases experiment dashboard
    ),
)

Unsloth: We found double BOS tokens - we shall remove one automatically.
Unsloth: We found double BOS tokens - we shall remove one automatically.


In [29]:
# 3. Apply Gemma 2 loss masking
trainer = train_on_responses_only(
    trainer,
    instruction_part="<start_of_turn>user\n",
    response_part="<start_of_turn>model\n",
)

Unsloth: Removed 632 out of 1514 samples from train_dataset where all labels were -100 (no response marker found, usually truncation). This prevents NaN loss during training.
Unsloth: Removed 171 out of 379 samples from eval_dataset where all labels were -100 (no response marker found, usually truncation). This prevents NaN loss during training.


In [30]:
# # Grab the first batch and decode
# batch = next(iter(trainer.get_train_dataloader()))
# input_ids = batch["input_ids"][0]
# labels = batch["labels"][0]

# # Inspect Gemma 2 boundary tokens specifically
# for tok, lab in zip(input_ids[:1024], labels[:1024]):
#     decoded = repr(tokenizer.decode([tok]))
#     mask_status = "MASKED (-100)" if lab == -100 else f"TRAIN ({lab})"
#     print(f"{decoded:<30} -> {mask_status}")

In [31]:
import os
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"

In [32]:
# ==========================================
# 7. EXECUTE TRAINING & RUN LOGGING
# ==========================================
print("Starting production training run...")
trainer_stats = trainer.train()
print("Training completed successfully!")

# Started on 17:11
# Started on 14:40 Nov 3

Starting production training run...


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 882 | Num Epochs = 9 | Total steps = 500
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 20,766,720 of 2,635,108,608 (0.79% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
50,0.018569,0.025120
100,0.026913,0.021507
150,0.004009,0.025217
200,0.000823,0.027892
250,0.000682,0.030985
300,0.000367,0.032818


Filter:   0%|          | 0/208 [00:00<?, ? examples/s]

Unsloth: Not an error, but `num_items_in_batch` could not be counted for Gemma2ForCausalLM's labels, so each micro-batch loss is a mean.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


KeyboardInterrupt: 

In [11]:
# ==========================================
# 8. POST-TRAINING INFERENCE VALIDATION
# ==========================================
# Switch model into native 2x faster inference mode utilizing Unsloth's optimized kernels
FastLanguageModel.for_inference(model)

Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Gemma2TextScaledWordEmbedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear4bit(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2304, bias=False)
          (rotary_emb): GemmaFixedRotaryEmbedding()
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear4bit(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear4bit(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear4bit(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_atte

In [33]:
from unsloth import FastLanguageModel

# Load the best checkpoint (around step 100 where eval/loss was lowest)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "outputs/checkpoint-100", # or checkpoint-150
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model) # Enable 2x faster inference

==((====))==  Unsloth 2026.10.11: Fast Gemma2 patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Unsloth: Will load outputs/checkpoint-100 as a legacy tokenizer.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma2ForCausalLM(
      (model): Gemma2Model(
        (embed_tokens): Gemma2TextScaledWordEmbedding(256000, 2304, padding_idx=0)
        (layers): ModuleList(
          (0-25): 26 x Gemma2DecoderLayer(
            (self_attn): Gemma2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2304, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2304, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
        

In [10]:
# Define your tool definitions (matching your training data format)
tools_definition = """
You have access to the following tools:
<tool>
{
  "name": "get_current_weather",
  "description": "Get the current weather for a given location",
  "parameters": {
    "type": "object",
    "properties": {
      "location": {"type": "string", "city": "The Name of the City"}
    },
    "required": ["location"]
  }
}
</tool>
"""

system_content = f"""You are a function calling AI. You MUST output a tool call whenever a user query can be answered using the available tools. Do not write conversational filler text; output only the tool invocation block.
{tools_definition}

Please tell me the current weather for the following cities.
1. Karachi 
2. Rawalpindi
3. Murree

Also I need to know the weather for capital of south korea, I am not able to recall the name though
"""

In [11]:
print(system_content)

You are a function calling AI. You MUST output a tool call whenever a user query can be answered using the available tools. Do not write conversational filler text; output only the tool invocation block.

You have access to the following tools:
<tool>
{
  "name": "get_current_weather",
  "description": "Get the current weather for a given location",
  "parameters": {
    "type": "object",
    "properties": {
      "location": {"type": "string", "city": "The Name of the City"}
    },
    "required": ["location"]
  }
}
</tool>


Please tell me the current weather for the following cities.
1. Karachi 
2. Rawalpindi
3. Murree

Also I need to know the weather for capital of south korea, I am not able to recall the name though



In [15]:
test_prompt = tokenizer.apply_chat_template([
    {"role": "user", "content": system_content},
    # {"role": "user", "content": "Get the current weather forecast for Tokyo, Japan."}
], tokenize=False, add_generation_prompt=True)

In [16]:
print(test_prompt)

<bos><start_of_turn>user
You are a function calling AI. You MUST output a tool call whenever a user query can be answered using the available tools. Do not write conversational filler text; output only the tool invocation block.

You have access to the following tools:
<tool>
{
  "name": "get_current_weather",
  "description": "Get the current weather for a given location",
  "parameters": {
    "type": "object",
    "properties": {
      "location": {"type": "string", "city": "The Name of the City"}
    },
    "required": ["location"]
  }
}
</tool>


Please tell me the current weather for the following cities.
1. Karachi 
2. Rawalpindi
3. Murree

Also I need to know the weather for capital of south korea, I am not able to recall the name though<end_of_turn>
<start_of_turn>model



In [18]:
# Tokenize and generate
inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")

In [85]:

outputs = model.generate(**inputs, max_new_tokens=512, use_cache=True, temperature=0.0)
print("\n--- INFERENCE TEST OUTPUT ---")
print(tokenizer.batch_decode(outputs)[0])

Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- INFERENCE TEST OUTPUT ---
<bos><bos><start_of_turn>user
You are a function calling AI. You MUST output a tool call whenever a user query can be answered using the available tools. Do not write conversational filler text; output only the tool invocation block.

You have access to the following tools:
<tool>
{
  "name": "get_current_weather",
  "description": "Get the current weather for a given location",
  "parameters": {
    "type": "object",
    "properties": {
      "location": {"type": "string", "city": "The Name of the City"}
    },
    "required": ["location"]
  }
}
</tool>


Please tell me the current weather for the following cities.
1. Karachi 
2. Rawalpindi
3. Murree

Also I need to know the weather for capital of south korea, I am not able to recall the name though<end_of_turn>
<start_of_turn>model
<tool>
{
  "name": "get_current_weather",
  "description": "Get the current weather for a given location",
  "parameters": {
    "type": "object",
    "properties": {
      "

In [7]:
# Load the model from HF
# Path to your Hugging Face model repository
model_id = "m-aamir2979/gemma-2-2b-bnb-4bit-fineTuned-tool-calling-adapter"

# 1. Load the model and tokenizer directly from HF Hub
fine_tuned_model, fine_tuned_tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_id,
    max_seq_length = 2048,
    dtype = None,        # Auto-detects (float16 for T4)
    load_in_4bit = True,  # Enable 4-bit quantization
    token=UserSecretsClient().get_secret("HF_TOKEN")
)

# 2. Activate 2x faster native inference mode
FastLanguageModel.for_inference(fine_tuned_model)


==((====))==  Unsloth 2026.10.11: Fast Gemma2 patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Unsloth: Will load unsloth/gemma-2-2b-it-bnb-4bit as a legacy tokenizer.
/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Unsloth: compiling each transformer block as one region (Gemma2DecoderLayer).


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma2ForCausalLM(
      (model): Gemma2Model(
        (embed_tokens): Gemma2TextScaledWordEmbedding(256000, 2304, padding_idx=0)
        (layers): ModuleList(
          (0-25): 26 x Gemma2DecoderLayer(
            (self_attn): Gemma2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2304, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2304, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
        

In [ ]:
test_prompt = fine_tuned_tokenizer.apply_chat_template([
    {"role": "user", "content": "Hello, Whats up bro"},
    # {"role": "user", "content": "Get the current weather forecast for Tokyo, Japan."}
], tokenize=False, add_generation_prompt=True)

In [ ]:
# Tokenize and generate
inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")

In [20]:
outputs = fine_tuned_model.generate(
    **inputs,
    max_new_tokens = 512,
    use_cache = True,
    temperature = 0.0,  # Low temperature for strict schema adherence
    do_sample = False
)

response = fine_tuned_tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
print(response)

Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


user
You are a function calling AI. You MUST output a tool call whenever a user query can be answered using the available tools. Do not write conversational filler text; output only the tool invocation block.

You have access to the following tools:
<tool>
{
  "name": "get_current_weather",
  "description": "Get the current weather for a given location",
  "parameters": {
    "type": "object",
    "properties": {
      "location": {"type": "string", "city": "The Name of the City"}
    },
    "required": ["location"]
  }
}
</tool>


Please tell me the current weather for the following cities.
1. Karachi 
2. Rawalpindi
3. Murree

Also I need to know the weather for capital of south korea, I am not able to recall the name though
model
<tool>
{
  "name": "get_current_weather",
  "description": "Get the current weather for a given location",
  "parameters": {
    "type": "object",
    "properties": {
      "location": {"type": "string", "city": "Karachi"}
    },
    "required": ["location"]